FastAPI 为何非常适合构建LLM的API

1. 无需繁琐仪式的异步优先
LLM 调用是 I/O 密集型的：需要通过网络跳转到提供商（或你的推理服务器）、向向量存储查询、从对象存储中获取数据。async 路由 + httpx.AsyncClient 可以让你并行处理调用、流式传输令牌，并在负载下保持可预测的延迟。
2. 自付其值的类型安全Pydantic 
模型强制执行请求/响应契约。你可以在用户之前发现形状不匹配（幻觉字段、缺少元数据）的问题。额外好处：你的 OpenAPI 规范自动生成且始终保持同步。
3. 合理的流式传输
无论你更喜欢服务器发送事件（SSE）还是 WebSockets，FastAPI 都让令牌流式传输变得简单。这对于聊天界面和仪表板的用户体验来说是黄金法则。
4. 重要的中间件
身份验证、速率限制、追踪和缓存可以干净利落地集成进来。你可以添加全局超时、记录传入/传出的有效载荷大小，或者注入相关 ID——而无需在路由代码中到处散落。
5. 适合生产环境的电池（以及合理的开发体验）
你得到了三合一：自动生成文档在 /docs，简单易用的依赖注入，以及友好的错误消息。你的团队可以更快地行动，同时减少出错。


In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
import httpx
import asyncio

app = FastAPI(title="LLM API", version="1.0")

class ChatRequest(BaseModel):
    messages: list[dict] = Field(..., description="OpenAI 风格的聊天格式")
    temperature: float = 0.2
    max_tokens: int = 512

async def llm_stream(req: ChatRequest):
    # 示例：代理到支持事件流的模型服务器
    timeout = httpx.Timeout(30.0, read=60.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        async with client.stream(
            "POST",
            "http://inference:8000/v1/chat/completions",
            json=req.model_dump(),
            headers={"x-trace-id": "inject-your-id"}
        ) as r:
            if r.status_code != 200:
                raise HTTPException(r.status_code, "上游错误")
            async for line in r.aiter_lines():
                if line:
                    yield f"data: {line}\n\n"
    # SSE 需要在某些客户端上干净地结束
    yield ": done\n\n"

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    return StreamingResponse(llm_stream(req), media_type="text/event-stream")

In [ ]:
from typing import AsyncIterator
from fastapi import APIRouter
from pydantic import BaseModel

router = APIRouter(prefix="/rag")

class RAGQuery(BaseModel):
    query: str
    k: int = 4

async def retrieve_chunks(q: str, k: int) -> list[str]:
    # 假设：异步向量搜索 + 对象存储获取
    await asyncio.sleep(0)  # 让出控制权
    return [f"chunk_{i}:{q}" for i in range(k)]

async def generate_stream(prompt: str) -> AsyncIterator[str]:
    # 假设：从你的模型流式传输令牌
    for token in ["Sure,", " here", " are", " the", " results."]:
        await asyncio.sleep(0.05)
        yield token

@router.post("/stream")
async def rag_stream(req: RAGQuery):
    async def sse():
        chunks = await retrieve_chunks(req.query, req.k)
        prompt = f"Use these:\n{chr(10).join(chunks)}\nQ: {req.query}\nA:"
        async for tok in generate_stream(prompt):
            yield f"data: {tok}\n\n"
        yield ": done\n\n"
    return StreamingResponse(sse(), media_type="text/event-stream")

app.include_router(router)

In [ ]:
import uuid, logging
from fastapi import Request
logger = logging.getLogger("uvicorn.access")

@app.middleware("http")
async def add_correlation_id(request: Request, call_next):
    cid = request.headers.get("x-correlation-id", str(uuid.uuid4()))
    response = await call_next(request)
    response.headers["x-correlation-id"] = cid
    logger.info("req",
        extra={"path": request.url.path, "cid": cid, "user": request.headers.get("x-user-id")}
    )
    return response

In [ ]:
import random
from fastapi import Request

RETRY_BACKOFF = [0.2, 0.5, 1.0]

async def with_retries(fn):
    for i, backoff in enumerate(RETRY_BACKOFF):
        try:
            return await fn()
        except httpx.RequestError:
            if i == len(RETRY_BACKOFF) - 1:
                raise
            await asyncio.sleep(backoff + random.random()/10)

封装上游调用，避免短暂的故障导致请求失败。保持低上限以保护 p99。输入大小限制和模式锁定拒绝超过 N 个令牌或超过 M KB 的负载。使用 Pydantic 验证器强制执行已知的工具/JSON 形状。版本化响应模式（v1、v2）以保持客户端稳定。真正有用的缓存提示/结果缓存，以 (模型, 消息哈希, 工具哈希) 为键。检索缓存，用于 RAG 块（TTL 5-30 分钟）。冷路径缓存，用于模型元数据和嵌入配置。不会在以后咬你的速率限制每用户、每组织和每 IP 的桶。返回 429 并附带 Retry-After。记录谁被限制——这可以发现滥用和增长。